# FinTrace — Forensic Transaction Analytics Platform

A standalone investigative layer built on top of the PaySim mobile-money
dataset. Where a fraud *detection* model answers "is this suspicious?",
FinTrace answers **who was involved, when did the behavior change, and
why was this flagged?** for any account under investigation.

**Run the cells top to bottom.** Section 7 demos everything on synthetic
data; swap in the real PaySim CSV (see Section 6) for actual results.

**Modules:**
1. Setup
2. Data Layer — loads PaySim, indexes by account
3. Graph Module — entity relationship graph + Louvain clustering
4. Timeline Module — chronological reconstruction + z-score flags
5. Explainability — rule-based, explainable anomaly scoring
6. Report Generator — renders the HTML case report
7. Demo — synthetic data, full pipeline, rendered case report


## 1. Setup

In [ ]:
!pip install -q python-louvain networkx jinja2

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

import pandas as pd
import numpy as np
import networkx as nx
import community as community_louvain  # python-louvain
from dataclasses import dataclass, field
from typing import Optional
from datetime import datetime, timezone
from pathlib import Path
from jinja2 import Template


## 2. Data Layer

Loads the PaySim transaction log and restructures it around **accounts
as first-class entities** instead of isolated transaction rows.

Expected PaySim schema (standard Kaggle *"Synthetic Financial Datasets
For Fraud Detection"* columns):

```
step, type, amount, nameOrig, oldbalanceOrg, newbalanceOrig,
nameDest, oldbalanceDest, newbalanceDest, isFraud, isFlaggedFraud
```

If your CSV uses different column names, pass a `column_map` to
`load_transactions` to rename them onto this schema.


In [ ]:
logger = logging.getLogger("fintrace.data_layer")

REQUIRED_COLUMNS = [
    "step", "type", "amount", "nameOrig", "oldbalanceOrg", "newbalanceOrig",
    "nameDest", "oldbalanceDest", "newbalanceDest", "isFraud",
]


def load_transactions(path: str, column_map: Optional[dict] = None,
                       nrows: Optional[int] = None) -> pd.DataFrame:
    """Load the raw PaySim CSV and validate/normalize its schema."""
    df = pd.read_csv(path, nrows=nrows)

    if column_map:
        df = df.rename(columns=column_map)

    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(
            f"Input data is missing required columns: {missing}. "
            f"Pass a column_map= to rename your columns onto the PaySim schema."
        )

    if "isFlaggedFraud" not in df.columns:
        df["isFlaggedFraud"] = 0

    # Synthetic, monotonically increasing timestamp derived from PaySim's
    # `step` field (each step = 1 simulated hour).
    df["timestamp"] = pd.to_datetime(df["step"], unit="h", origin="2023-01-01")

    df = df.reset_index(drop=True)
    df["txn_id"] = df.index.map(lambda i: f"TXN{i:08d}")

    logger.info("Loaded %d transactions covering %d origin accounts.",
                len(df), df["nameOrig"].nunique())
    return df


@dataclass
class AccountRecord:
    """All known activity for a single account, from both sides of the ledger."""
    account_id: str
    outgoing: pd.DataFrame = field(repr=False)
    incoming: pd.DataFrame = field(repr=False)

    @property
    def all_activity(self) -> pd.DataFrame:
        """Outgoing + incoming transactions, sorted chronologically."""
        out = self.outgoing.copy()
        out["direction"] = "outgoing"
        out["counterparty"] = out["nameDest"]

        inc = self.incoming.copy()
        inc["direction"] = "incoming"
        inc["counterparty"] = inc["nameOrig"]

        cols = ["txn_id", "timestamp", "step", "type", "amount", "direction",
                "counterparty", "oldbalanceOrg", "newbalanceOrig",
                "oldbalanceDest", "newbalanceDest", "isFraud"]
        combined = pd.concat([out[cols], inc[cols]], ignore_index=True)
        return combined.sort_values("timestamp").reset_index(drop=True)

    @property
    def counterparties(self):
        return sorted(set(self.outgoing["nameDest"]) | set(self.incoming["nameOrig"]))

    @property
    def total_sent(self) -> float:
        return float(self.outgoing["amount"].sum())

    @property
    def total_received(self) -> float:
        return float(self.incoming["amount"].sum())

    @property
    def flagged_transaction_count(self) -> int:
        return int(self.outgoing["isFraud"].sum() + self.incoming["isFraud"].sum())


class AccountIndex:
    """Fast lookup of an account's full transaction history, in both
    directions, built once over the full transaction log."""

    def __init__(self, df: pd.DataFrame):
        self._df = df
        self._by_orig = {k: v for k, v in df.groupby("nameOrig")}
        self._by_dest = {k: v for k, v in df.groupby("nameDest")}
        self._all_accounts = sorted(
            set(self._by_orig.keys()) | set(self._by_dest.keys())
        )

    def __len__(self):
        return len(self._all_accounts)

    def __contains__(self, account_id):
        return account_id in self._all_accounts

    @property
    def accounts(self):
        return self._all_accounts

    def get(self, account_id: str) -> AccountRecord:
        empty = self._df.iloc[0:0]
        outgoing = self._by_orig.get(account_id, empty)
        incoming = self._by_dest.get(account_id, empty)
        if outgoing.empty and incoming.empty:
            raise KeyError(f"Unknown account: {account_id}")
        return AccountRecord(account_id=account_id, outgoing=outgoing, incoming=incoming)

    def flagged_accounts(self):
        """Accounts on either side of at least one fraud-labeled transaction —
        the natural starting point for an investigation."""
        fraud_txns = self._df[self._df["isFraud"] == 1]
        return sorted(set(fraud_txns["nameOrig"]) | set(fraud_txns["nameDest"]))


def build_account_index(df: pd.DataFrame) -> AccountIndex:
    return AccountIndex(df)


## 3. Graph Module

Builds the entity relationship graph: accounts as nodes, transactions as
weighted, directed edges. Applies **Louvain community detection** to
surface clusters such as money-mule chains and circular transfer rings
that are invisible in a flat transaction table.


In [ ]:
logger = logging.getLogger("fintrace.graph_module")


def build_graph(df: pd.DataFrame, min_amount: float = 0.0) -> nx.DiGraph:
    """Each edge (u -> v) is weighted by total amount transferred from u to
    v, and carries a `count` and a `fraud_count`."""
    G = nx.DiGraph()
    txns = df[df["amount"] >= min_amount]
    for row in txns.itertuples(index=False):
        u, v = row.nameOrig, row.nameDest
        if G.has_edge(u, v):
            G[u][v]["weight"] += row.amount
            G[u][v]["count"] += 1
            G[u][v]["fraud_count"] += int(row.isFraud)
        else:
            G.add_edge(u, v, weight=row.amount, count=1, fraud_count=int(row.isFraud))

    logger.info("Built graph with %d accounts and %d transaction edges.",
                G.number_of_nodes(), G.number_of_edges())
    return G


def detect_communities(G: nx.DiGraph) -> dict:
    """Louvain community detection over the graph's undirected projection."""
    undirected = G.to_undirected()
    partition = community_louvain.best_partition(undirected, weight="weight")
    logger.info("Detected %d communities across %d accounts.",
                len(set(partition.values())), len(partition))
    return partition


def community_risk_scores(G: nx.DiGraph, partition: dict) -> pd.DataFrame:
    """Rank each community by its share of fraud-labeled edges."""
    rows = []
    communities = {}
    for account, comm_id in partition.items():
        communities.setdefault(comm_id, []).append(account)

    for comm_id, members in communities.items():
        member_set = set(members)
        sub_edges = [(u, v, d) for u, v, d in G.edges(data=True)
                     if u in member_set and v in member_set]
        total_txns = sum(d["count"] for _, _, d in sub_edges)
        fraud_txns = sum(d["fraud_count"] for _, _, d in sub_edges)
        total_amount = sum(d["weight"] for _, _, d in sub_edges)
        rows.append({
            "community_id": comm_id, "size": len(members),
            "internal_transactions": total_txns,
            "internal_fraud_transactions": fraud_txns,
            "fraud_rate": (fraud_txns / total_txns) if total_txns else 0.0,
            "total_internal_amount": total_amount,
        })

    return pd.DataFrame(rows).sort_values(
        ["fraud_rate", "total_internal_amount"], ascending=False
    ).reset_index(drop=True)


def get_neighborhood(G: nx.DiGraph, account_id: str, depth: int = 1) -> nx.DiGraph:
    """Induced subgraph of an account's ego network out to `depth` hops."""
    if account_id not in G:
        raise KeyError(f"Account {account_id} is not present in the graph.")
    nodes = {account_id}
    frontier = {account_id}
    undirected = G.to_undirected()
    for _ in range(depth):
        next_frontier = set()
        for node in frontier:
            next_frontier.update(undirected.neighbors(node))
        nodes.update(next_frontier)
        frontier = next_frontier
    return G.subgraph(nodes).copy()


def find_circular_transfers(G: nx.DiGraph, max_length: int = 4):
    """Detect simple cycles up to `max_length` nodes — a classic
    money-laundering pattern (funds routed back to their origin)."""
    cycles = []
    try:
        for cycle in nx.simple_cycles(G, length_bound=max_length):
            if len(cycle) > 1:
                cycles.append(cycle)
    except TypeError:
        for cycle in nx.simple_cycles(G):
            if 1 < len(cycle) <= max_length:
                cycles.append(cycle)
    return cycles


def fan_in_fan_out(G: nx.DiGraph, in_threshold: int = 5, out_threshold: int = 5) -> dict:
    """Flag accounts with unusually high fan-in/fan-out — a common
    signature of collection/distribution "mule" accounts."""
    flagged = {}
    for node in G.nodes():
        in_deg, out_deg = G.in_degree(node), G.out_degree(node)
        if in_deg >= in_threshold or out_deg >= out_threshold:
            flagged[node] = {
                "in_degree": in_deg, "out_degree": out_deg,
                "pattern": "fan-in" if in_deg >= in_threshold and in_deg > out_deg
                else "fan-out" if out_deg >= out_threshold else "fan-in",
            }
    return flagged


## 4. Timeline Module

Reconstructs a chronological view of a single account's activity and
flags **moments of interest**: statistically unusual points in that
account's *own* history (not a global threshold) — e.g. a long-dormant
account suddenly transferring a large sum.


In [ ]:
logger = logging.getLogger("fintrace.timeline_module")


@dataclass
class TimelineEvent:
    txn_id: str
    timestamp: pd.Timestamp
    step: int
    direction: str
    counterparty: str
    amount: float
    balance_after: float
    z_score: float
    is_flagged_moment: bool
    flag_reasons: list


def build_timeline(account_id: str, index: AccountIndex) -> pd.DataFrame:
    """Account's full activity, chronologically ordered, with a running
    balance and a z-score vs. this account's *own* historical mean/std."""
    record = index.get(account_id)
    activity = record.all_activity.copy()
    if activity.empty:
        return activity

    def resolve_balance(row):
        return row["newbalanceOrig"] if row["direction"] == "outgoing" else row["newbalanceDest"]

    activity["balance_after"] = activity.apply(resolve_balance, axis=1)

    mean_amt = activity["amount"].mean()
    std_amt = activity["amount"].std(ddof=0)
    activity["z_score"] = 0.0 if (std_amt == 0 or np.isnan(std_amt)) \
        else (activity["amount"] - mean_amt) / std_amt

    activity["hours_since_prev"] = activity["step"].diff()
    return activity


def flag_moments_of_interest(timeline: pd.DataFrame, z_threshold: float = 2.5,
                              dormancy_hours: int = 168):
    """dormancy_hours defaults to 168 (one simulated week, step==hour)."""
    events = []
    if timeline.empty:
        return events

    for row in timeline.itertuples(index=False):
        reasons = []
        if abs(row.z_score) >= z_threshold:
            reasons.append(f"Amount is {row.z_score:+.1f} standard deviations from "
                            f"this account's typical transaction size.")

        gap = getattr(row, "hours_since_prev", np.nan)
        if not np.isnan(gap) and gap >= dormancy_hours and abs(row.z_score) >= 1.0:
            reasons.append(f"Follows {gap:.0f} hours of inactivity — a "
                            f"dormant-then-active pattern.")

        if getattr(row, "isFraud", 0) == 1:
            reasons.append("Transaction is labeled fraudulent in the source data.")

        events.append(TimelineEvent(
            txn_id=row.txn_id, timestamp=row.timestamp, step=row.step,
            direction=row.direction, counterparty=row.counterparty,
            amount=row.amount, balance_after=row.balance_after,
            z_score=row.z_score, is_flagged_moment=len(reasons) > 0,
            flag_reasons=reasons,
        ))

    n_flagged = sum(e.is_flagged_moment for e in events)
    logger.info("Timeline built with %d events, %d flagged as moments of interest.",
                len(events), n_flagged)
    return events


## 5. Explainability

Rule-based, explainable anomaly scoring. Deliberately avoids an opaque
ML classifier here: FinTrace's job is to **justify a flag to a human
investigator**, not just produce a probability. Every score comes with a
plain-language breakdown of which signals drove it.


In [ ]:
logger = logging.getLogger("fintrace.explainability")


@dataclass
class AnomalyExplanation:
    account_id: str
    risk_score: float
    risk_band: str
    signals: list = field(default_factory=list)

    def summary(self) -> str:
        if not self.signals:
            return "No significant anomaly signals detected for this account."
        top = sorted(self.signals, key=lambda s: -s["weight"])[0]
        return f"Primary driver: {top['detail']}"


def _iqr_outlier_fraction(values: pd.Series) -> float:
    if len(values) < 4:
        return 0.0
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    if iqr == 0:
        return 0.0
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = values[(values < lower) | (values > upper)]
    return len(outliers) / len(values)


def score_account(account_id: str, index: AccountIndex,
                   new_counterparty_window: int = 5) -> AnomalyExplanation:
    """Explainable anomaly score in [0, 100], combining:
      - z-score deviation from the account's own typical amount
      - IQR outlier fraction
      - counterparty novelty in recent activity
      - direct fraud labels
    """
    record = index.get(account_id)
    activity = record.all_activity
    signals, score = [], 0.0

    if activity.empty:
        return AnomalyExplanation(account_id, 0.0, "Low", [])

    amounts = activity["amount"]
    mean_amt, std_amt = amounts.mean(), amounts.std(ddof=0)

    # Signal 1: extreme z-score transactions
    extreme_frac = float(((amounts - mean_amt) / std_amt).abs().ge(2.5).mean()) \
        if std_amt and not np.isnan(std_amt) else 0.0
    if extreme_frac > 0:
        weight = min(40.0, extreme_frac * 100)
        score += weight
        signals.append({"feature": "amount_zscore",
            "detail": f"{extreme_frac:.0%} of this account's transactions are more "
                      f"than 2.5 standard deviations from its typical amount.",
            "weight": weight})

    # Signal 2: IQR-based outlier share
    outlier_frac = _iqr_outlier_fraction(amounts)
    if outlier_frac > 0:
        weight = min(25.0, outlier_frac * 60)
        score += weight
        signals.append({"feature": "iqr_outliers",
            "detail": f"{outlier_frac:.0%} of transaction amounts fall outside the "
                      f"account's normal interquartile range.",
            "weight": weight})

    # Signal 3: counterparty novelty in recent activity
    ordered = activity.sort_values("timestamp")
    seen, novel_recent = set(), 0
    recent = ordered.tail(new_counterparty_window)
    recent_ids = set(recent["txn_id"])
    for _, row in ordered.iterrows():
        if row["counterparty"] not in seen and row["txn_id"] in recent_ids:
            novel_recent += 1
        seen.add(row["counterparty"])
    if len(recent) > 0:
        novelty_frac = novel_recent / len(recent)
        if novelty_frac >= 0.4:
            weight = min(20.0, novelty_frac * 30)
            score += weight
            signals.append({"feature": "counterparty_novelty",
                "detail": f"{novel_recent} of the last {len(recent)} transactions are "
                          f"with counterparties never seen before in this account's history.",
                "weight": weight})

    # Signal 4: direct fraud labels
    fraud_count = int(activity["isFraud"].sum())
    if fraud_count > 0:
        weight = min(35.0, fraud_count * 15)
        score += weight
        signals.append({"feature": "labeled_fraud",
            "detail": f"{fraud_count} transaction(s) on this account are directly "
                      f"labeled fraudulent in the source data.",
            "weight": weight})

    score = min(100.0, score)
    band = ("Critical" if score >= 75 else "High" if score >= 50
            else "Medium" if score >= 25 else "Low")

    logger.info("Scored account %s: %.1f (%s) from %d signal(s).",
                account_id, score, band, len(signals))
    return AnomalyExplanation(account_id=account_id, risk_score=round(score, 1),
                               risk_band=band, signals=signals)


## 6. Report Generator

Pulls together the data layer, graph module, timeline module, and
explainability scoring into a single auto-generated case report per
investigated account — styled navy/gold to match the rest of the
project deliverables. The template is embedded as a string here (rather
than a separate `.j2` file) so the whole thing runs in one notebook.


In [ ]:
CASE_REPORT_TEMPLATE = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>FinTrace Case Report — {{ account_id }}</title>
<style>
  :root {
    --navy: #1E2761;
    --navy-dark: #161C4A;
    --ice: #CADCFC;
    --gold: #D4AF37;
    --slate: #4A5578;
    --offwhite: #F7F8FC;
  }
  * { box-sizing: border-box; }
  body {
    font-family: "Calibri", "Segoe UI", Arial, sans-serif;
    background: var(--offwhite);
    color: var(--navy);
    margin: 0;
    padding: 0;
  }
  .header {
    background: var(--navy-dark);
    color: white;
    padding: 36px 48px;
  }
  .header .eyebrow {
    color: var(--gold);
    letter-spacing: 3px;
    font-size: 12px;
    font-weight: bold;
    text-transform: uppercase;
  }
  .header h1 {
    font-family: Cambria, Georgia, serif;
    font-size: 34px;
    margin: 8px 0 4px 0;
  }
  .header .meta {
    color: var(--ice);
    font-size: 14px;
  }
  .risk-band {
    display: inline-block;
    margin-top: 14px;
    padding: 6px 16px;
    border-radius: 20px;
    font-weight: bold;
    font-size: 13px;
  }
  .risk-Low { background: #3a4a2f; color: #b7e39a; }
  .risk-Medium { background: #4a3f1f; color: var(--gold); }
  .risk-High { background: #4a241f; color: #f2a679; }
  .risk-Critical { background: #5a1f24; color: #ff9d9d; }

  .content { padding: 32px 48px; max-width: 900px; margin: 0 auto; }
  .card {
    background: white;
    border: 1px solid var(--ice);
    border-radius: 10px;
    padding: 22px 26px;
    margin-bottom: 22px;
    box-shadow: 0 3px 10px rgba(30, 39, 97, 0.08);
  }
  .card h2 {
    font-family: Cambria, Georgia, serif;
    font-size: 19px;
    margin: 0 0 14px 0;
    color: var(--navy);
    border-bottom: 2px solid var(--gold);
    padding-bottom: 8px;
    display: inline-block;
  }
  .stat-grid {
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 14px;
    margin-top: 6px;
  }
  .stat { background: var(--offwhite); border-radius: 8px; padding: 12px 14px; }
  .stat .label { font-size: 11px; color: var(--slate); text-transform: uppercase; letter-spacing: 1px;}
  .stat .value { font-size: 20px; font-weight: bold; color: var(--navy); margin-top: 4px;}

  table { width: 100%; border-collapse: collapse; font-size: 13.5px; }
  th { text-align: left; color: var(--slate); font-size: 11px; text-transform: uppercase;
       letter-spacing: 0.5px; padding: 6px 8px; border-bottom: 2px solid var(--ice); }
  td { padding: 8px; border-bottom: 1px solid var(--offwhite); vertical-align: top; }
  tr.flagged { background: #fff6ea; }
  .badge { display: inline-block; padding: 2px 8px; border-radius: 10px; font-size: 11px; font-weight: bold; }
  .badge-out { background: #eef1fb; color: var(--navy); }
  .badge-in { background: #e9f7ef; color: #1e6b3e; }

  .signal { padding: 10px 0; border-bottom: 1px solid var(--offwhite); }
  .signal:last-child { border-bottom: none; }
  .signal .bar-bg { background: var(--offwhite); border-radius: 6px; height: 8px; margin-top: 6px; overflow: hidden; }
  .signal .bar-fill { background: var(--gold); height: 100%; }
  .signal .detail { font-size: 13.5px; color: var(--slate); margin-top: 4px; }

  .neighbor-list span { display: inline-block; background: var(--offwhite); color: var(--navy);
    border: 1px solid var(--ice); border-radius: 14px; padding: 4px 12px; margin: 3px 4px 3px 0; font-size: 12.5px;}

  .footer { text-align: center; color: var(--slate); font-size: 11.5px; padding: 24px; }
</style>
</head>
<body>

  <div class="header">
    <div class="eyebrow">FinTrace &middot; Case Report</div>
    <h1>Account {{ account_id }}</h1>
    <div class="meta">Generated {{ generated_at }} &middot; {{ total_transactions }} transactions on record</div>
    <div class="risk-band risk-{{ risk_band }}">{{ risk_band|upper }} RISK &middot; SCORE {{ risk_score }}/100</div>
  </div>

  <div class="content">

    <div class="card">
      <h2>Account Summary</h2>
      <div class="stat-grid">
        <div class="stat"><div class="label">Total Sent</div><div class="value">{{ total_sent }}</div></div>
        <div class="stat"><div class="label">Total Received</div><div class="value">{{ total_received }}</div></div>
        <div class="stat"><div class="label">Counterparties</div><div class="value">{{ counterparty_count }}</div></div>
        <div class="stat"><div class="label">Fraud-Labeled Txns</div><div class="value">{{ fraud_count }}</div></div>
      </div>
    </div>

    <div class="card">
      <h2>Why This Account Was Flagged</h2>
      {% if signals %}
        {% for s in signals %}
        <div class="signal">
          <strong>{{ s.feature.replace('_',' ')|title }}</strong> &mdash; weight {{ s.weight|round(1) }}
          <div class="bar-bg"><div class="bar-fill" style="width: {{ (s.weight / 40 * 100) | round(0) }}%;"></div></div>
          <div class="detail">{{ s.detail }}</div>
        </div>
        {% endfor %}
      {% else %}
        <p>No significant anomaly signals detected for this account.</p>
      {% endif %}
    </div>

    <div class="card">
      <h2>Network Neighborhood</h2>
      <p style="color: var(--slate); font-size: 13.5px; margin-top: -6px;">
        Directly connected accounts (1-hop neighborhood in the entity graph){% if community_id is not none %}, community #{{ community_id }}{% endif %}.
      </p>
      <div class="neighbor-list">
        {% for n in neighbors %}<span>{{ n }}</span>{% endfor %}
      </div>
    </div>

    <div class="card">
      <h2>Activity Timeline</h2>
      <table>
        <tr>
          <th>Time (step)</th><th>Direction</th><th>Counterparty</th>
          <th>Amount</th><th>Balance After</th><th>Z-score</th><th>Notes</th>
        </tr>
        {% for e in timeline_events %}
        <tr class="{{ 'flagged' if e.is_flagged_moment else '' }}">
          <td>{{ e.step }}</td>
          <td><span class="badge {{ 'badge-out' if e.direction == 'outgoing' else 'badge-in' }}">{{ e.direction }}</span></td>
          <td>{{ e.counterparty }}</td>
          <td>{{ "%.2f"|format(e.amount) }}</td>
          <td>{{ "%.2f"|format(e.balance_after) }}</td>
          <td>{{ "%.2f"|format(e.z_score) }}</td>
          <td>{{ e.flag_reasons|join(' ') if e.flag_reasons else '' }}</td>
        </tr>
        {% endfor %}
      </table>
    </div>

  </div>

  <div class="footer">FinTrace &middot; Forensic Transaction Analytics Platform &middot; Generated automatically from PaySim transaction data</div>

</body>
</html>
"""

logger = logging.getLogger("fintrace.report_generator")


def generate_case_report(account_id: str, index: AccountIndex, G: nx.DiGraph,
                          output_path: str, partition: dict = None) -> str:
    """Build and write the full case report for one account. Returns the
    path the report was written to."""
    record = index.get(account_id)
    explanation = score_account(account_id, index)

    timeline_df = build_timeline(account_id, index)
    timeline_events = flag_moments_of_interest(timeline_df)

    try:
        neighborhood = get_neighborhood(G, account_id, depth=1)
        neighbors = sorted(n for n in neighborhood.nodes() if n != account_id)
    except KeyError:
        neighbors = []

    community_id = partition.get(account_id) if partition else None

    html = Template(CASE_REPORT_TEMPLATE).render(
        account_id=account_id,
        generated_at=datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC"),
        total_transactions=len(record.all_activity),
        risk_score=explanation.risk_score,
        risk_band=explanation.risk_band,
        signals=explanation.signals,
        total_sent=f"{record.total_sent:,.2f}",
        total_received=f"{record.total_received:,.2f}",
        counterparty_count=len(record.counterparties),
        fraud_count=record.flagged_transaction_count,
        neighbors=neighbors,
        community_id=community_id,
        timeline_events=timeline_events,
    )

    out = Path(output_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(html, encoding="utf-8")
    logger.info("Case report for %s written to %s", account_id, out)
    return str(out)


## 7. Load Your Dataset

This section works with **your own file** instead of synthetic data.
Run the cells in order — the first one just previews your columns so
you know whether you need a `column_map` before loading it properly.


### 7a. Upload your file

If you're in Colab and haven't uploaded it yet, run the cell below. If
you already uploaded it (or you're running locally), skip this cell and
just set `DATA_PATH` in the next one.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd

DATA_PATH = '/content/drive/MyDrive/Data analytics-dataset.csv'

df = pd.read_csv(DATA_PATH)



Mounted at /content/drive


### 7b. Preview your columns

Before loading through `load_transactions` (which validates the schema),
take a look at what you actually have.


In [ ]:
preview = pd.read_csv(DATA_PATH, nrows=5)
print("Columns in your file:")
print(list(preview.columns))
print()
preview


Columns in your file:
['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']



,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


### 7c. Map your columns (only if needed) and load

`load_transactions` expects these column names:

```
step, type, amount, nameOrig, oldbalanceOrg, newbalanceOrig,
nameDest, oldbalanceDest, newbalanceDest, isFraud
```

- **If your columns already match** (this is the case for the real
  Kaggle PaySim CSV), leave `COLUMN_MAP = {}` as-is.
- **If your names differ**, fill in `COLUMN_MAP` as
  `{"your_column_name": "expected_name", ...}` using the list you just
  printed above. You only need to map the columns that differ —
  anything already matching can be left out.
- **If a required column doesn't exist in your data at all**
  (e.g. no `isFraud` label), see the note below the code cell.


**If a required column is genuinely missing** from your dataset
(most commonly `oldbalanceOrg`/`newbalanceOrig`/`oldbalanceDest`/
`newbalanceDest`, or `isFraud`), you have two options:

1. **Balance columns missing**: fill them with `0` as a placeholder —
   the pipeline still runs, but the Timeline module's `balance_after`
   column and any balance-based signals won't be meaningful.
   ```python
   for col in ["oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest"]:
       if col not in df.columns:
           df[col] = 0.0
   ```
2. **`isFraud` missing** (you don't have ground-truth labels): set it
   to all zeros. FinTrace still works — the anomaly scoring in Section
   5 doesn't require fraud labels at all (only the "direct fraud
   labels" signal does), so accounts get ranked purely on statistical
   and network behavior.
   ```python
   if "isFraud" not in df.columns:
       df["isFraud"] = 0
   ```
   Run whichever of these you need **before** `load_transactions`, then
   re-run cell 7c.


### 7d. Build the entity graph

In [ ]:
for col in ["oldbalanceOrg","newbalanceOrig","oldbalanceDest","newbalanceDest"]:
  if col not in df.columns:
    df[col]=0.0

In [ ]:
if "isFraud" not in df.columns:
    df["isFraud"] = 0

In [ ]:
G = build_graph(df)
partition = detect_communities(G)


### 7e. Scan — rank the highest-priority accounts to investigate

In [ ]:
index = build_account_index(df)
flagged = index.flagged_accounts()

# If isFraud is all zeros (no ground-truth labels), flagged_accounts()
# will be empty — fall back to scoring every account instead.
candidates = flagged if flagged else index.accounts

scored = [score_account(acc, index) for acc in candidates]
scored.sort(key=lambda e: -e.risk_score)

print(f"Top {min(10, len(scored))} accounts by anomaly score:\n")
print(f"{'Account':<18}{'Score':>8}  {'Band':<10} Primary signal")
print("-" * 90)
for e in scored[:10]:
    print(f"{e.account_id:<18}{e.risk_score:>8.1f}  {e.risk_band:<10} {e.summary()}")

print("\nCommunity risk ranking (top 5):")
community_risk_scores(G, partition).head(5)

KeyError: "['txn_id', 'timestamp'] not in index"

### 7f. Generate and view a case report

In [ ]:
from IPython.display import HTML, display

# Defaults to the top-scored account from the scan above — override with
# a specific account ID if you want to investigate a different one.
ACCOUNT_TO_INVESTIGATE = scored[0].account_id if scored else index.accounts[0]
print(f"Investigating: {ACCOUNT_TO_INVESTIGATE}")

report_path = generate_case_report(ACCOUNT_TO_INVESTIGATE, index, G,
                                    f"output/case_{ACCOUNT_TO_INVESTIGATE}.html",
                                    partition=partition)
print(f"Case report written to: {report_path}")

display(HTML(Path(report_path).read_text()))


### 7g. (Optional) Download the report locally

In [ ]:
try:
    from google.colab import files
    files.download(report_path)
except ImportError:
    print("Not running in Colab — file is already saved at:", report_path)


---
## Appendix: Synthetic Demo Data

Not needed if Section 7 already worked on your real dataset. This is
only here so you can sanity-check the pipeline against known, seeded
patterns (a mule account, a circular transfer ring, a dormant account)
if something in Section 7 looks off and you want to confirm the modules
themselves are behaving correctly.


In [ ]:
import random, os

random.seed(42)
np.random.seed(42)

N_NORMAL_ACCOUNTS = 60
N_STEPS = 30 * 24
TYPES = ["PAYMENT", "TRANSFER", "CASH_OUT", "CASH_IN", "DEBIT"]

def account_id(prefix, i):
    return f"{prefix}{i:09d}"

rows = []
normal_accounts = [account_id("C", i) for i in range(N_NORMAL_ACCOUNTS)]
balances = {a: random.uniform(1000, 20000) for a in normal_accounts}

def txn(step, orig, dest, amount, is_fraud=0):
    old_orig = balances.get(orig, random.uniform(1000, 20000))
    new_orig = max(0.0, old_orig - amount)
    old_dest = balances.get(dest, random.uniform(0, 5000))
    new_dest = old_dest + amount
    balances[orig], balances[dest] = new_orig, new_dest
    rows.append({"step": step, "type": random.choice(TYPES), "amount": round(amount, 2),
                 "nameOrig": orig, "oldbalanceOrg": round(old_orig, 2),
                 "newbalanceOrig": round(new_orig, 2), "nameDest": dest,
                 "oldbalanceDest": round(old_dest, 2), "newbalanceDest": round(new_dest, 2),
                 "isFraud": is_fraud})

for step in range(N_STEPS):
    for _ in range(3):
        o, d = random.sample(normal_accounts, 2)
        txn(step, o, d, np.random.gamma(2, 80))

mule = account_id("C", 900)
senders = [account_id("C", 700 + i) for i in range(8)]
for i, s in enumerate(senders):
    txn(step=200 + i, orig=s, dest=mule, amount=random.uniform(4000, 9000))
for i in range(6):
    txn(step=210 + i, orig=mule, dest=account_id("C", 950 + i), amount=random.uniform(3000, 8000))

a, b, c = account_id("C", 800), account_id("C", 801), account_id("C", 802)
for i, (u, v) in enumerate([(a, b), (b, c), (c, a)]):
    txn(step=300 + i * 2, orig=u, dest=v, amount=15000, is_fraud=1)

dormant = account_id("C", 999)
balances[dormant] = 500.0
txn(step=1, orig=account_id("C", 1), dest=dormant, amount=100)
txn(step=N_STEPS - 5, orig=dormant, dest=account_id("C", 998), amount=45000, is_fraud=1)

df_synth = pd.DataFrame(rows).sort_values("step").reset_index(drop=True)
df_synth["isFlaggedFraud"] = 0

os.makedirs("data", exist_ok=True)
SYNTH_PATH = "data/sample_paysim.csv"
df_synth.to_csv(SYNTH_PATH, index=False)
print(f"Wrote {len(df_synth)} synthetic transactions to {SYNTH_PATH}")
print(f"Seeded accounts of interest: mule={mule}, ring=({a},{b},{c}), dormant={dormant}")
print("Re-run Section 7 with DATA_PATH = SYNTH_PATH and COLUMN_MAP = {} to test against this.")


## Design notes

- **Explainability over accuracy**: anomaly scoring is deliberately
  rule-based (z-score / IQR deviation from an account's own history,
  counterparty novelty, direct fraud labels) rather than an opaque ML
  classifier, so every flag comes with a plain-language reason.
- **Per-account baselines**: z-scores are computed against each
  account's own transaction history, not a global threshold.
- **Louvain community detection** runs on the graph's undirected
  projection to surface clusters (mule rings, laundering chains);
  direction is preserved separately in the directed graph for trail
  reconstruction and cycle detection (`find_circular_transfers`).
- **Works without fraud labels**: if your dataset has no `isFraud`
  column, the scan step falls back to scoring every account instead of
  only pre-labeled ones — FinTrace's anomaly signals are statistical
  and network-based, not dependent on ground truth.

## Suggested next steps (stretch goals)
- Interactive dashboard (Streamlit/React) wrapping this pipeline with a
  search box and live graph visualization
- PDF export of the case report alongside HTML
- Batch report generation for every flagged account in one run
